## 1、TextSplitter的使用

1、使用细节 \
1:TextSplitter作为各种具体的文档拆分器的父类 \
2:内部定义了一些常用的属性： \
chunk_size:返回块的最大尺寸，单位是字符数。默认值为4000（由长度函数测量）\
chunk_overlap:相邻两个块之间的字符重叠数，避免信息在边界处被切断而丢失。默认值为200，通常会设置为chunk_size的10%-20% \
length_function:用于测量给定字符块中保留分隔数，默认值为False \
keep_separator:是否在块中保留分隔符，默认值为False \
add_start_index:如果为True，则在元数据中包含块的起始索引。默认值为False \
strip_whitespace:如果为True，则从每个文档的开始和结束处去除空白字符。默认值为True \

内部定义的常用方法：\
情况1:按照字符串进行拆分：\
split_text(xxx)：传入的参数类型：<font color='red'>字符串</font>；返回值的类型：list[str] \
create_documents(xxx):传入的参数类型：<font color='red'>list[str]</font>;返回值的类型：list[Document]。底层调用了split_text(xxx)

情况2:按照Document对象进行拆分 \
split_documents(xxx):传入的参数类型：<font color='red'>list[Document]</font>;返回值的类型：list[Document]。底册调用create_documents(xxx)

2、Document对象和Str是什么关系？ \
文档拆分器可以按照字符串进行切分，也可以按照Document进行切分。其中，Str可以理解为是Document对象的page_content属性

### 具体的拆分器1:CharacterTextSplitter。Split by character

In [8]:
# 举例1:体会chunk_size、chunk_overlap
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter

# 字符串示例 一个空格也是一个字符数
text = """
LangChain是一个用于开发语言模型驱动的应用程序的框架。它提供块一套工具和抽象方法，使开发者，
能够更容易地构建复杂的应用程序
"""
textSplitter=CharacterTextSplitter(
    # 若必须禁用分隔符，则实际块长略小于chunk_size（相当于默认了空格）
    separator='',  # 设置为空字符串时，表示禁用分隔符优先
    chunk_size=50, # 每块大小
    chunk_overlap=5, # 块与块之间的重复字符数
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度49
LangChain是一个用于开发语言模型驱动的应用程序的框架。它提供块一套工具和抽象方法，使开发者
--------------------------------------------------
块2:长度22
，使开发者，
能够更容易地构建复杂的应用程序
--------------------------------------------------


In [45]:
# 举例2：体会separator
from langchain_text_splitters import CharacterTextSplitter

# 字符串示例 一个空格也是一个字符数
text = """
这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分隔给予字符数。
"""
"""流程：
    1、先按separator分隔
    2、发现前2个块小于chunk_size的值就合并，如果大于，第一个块就不会并，接着看第二块和第三块
"""

textSplitter=CharacterTextSplitter(
    # 若必须禁用分隔符，则实际块长略小于chunk_size（相当于默认了空格）
    separator='。',  # 按句号分隔（分隔符优先）
    # 使用30的时候报错Created a chunk of size 33, which is longer than the specified 30
    # chunk_size=30, # 每块大小

    # 会把2个块合并成一个
    # chunk_size=44, # 每块大小

    # 会把后面2个块合并
    chunk_size=42, # 每块大小

    chunk_overlap=0, # 块与块之间的重复字符数
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度9
这是一个示例文本啊
--------------------------------------------------
块2:长度41
我们将使用CharacterTextSplitter将其分割成小块。分隔给予字符数
--------------------------------------------------


In [57]:
# 举例3:体会chunk_overlap
from langchain_text_splitters import CharacterTextSplitter

'''流程：
    1、先按separator分隔
    2、因为第一块+第二块小于chunk_size，则把1和2合并
    3、再看第二块是否小于chunk_overlap，不小于就不会有重叠部分，小于则把2和3合并
'''
# 2.自定义要分隔的文本
text='这是第一段文本。这是第二段文本。最后一段结束'

textSplitter=CharacterTextSplitter(
    separator='。',
    chunk_size=20,
    chunk_overlap=5,
    keep_separator=True # chunk中是否保留切割符
)
# 分隔文本
texts=textSplitter.split_text(text)
# print(texts)

for i, chunk in enumerate(texts):
    print(f'块{i+1}:长度{len(chunk)}')
    print(chunk)
    print('-'*50)

块1:长度15
这是第一段文本。这是第二段文本
--------------------------------------------------
块2:长度7
。最后一段结束
--------------------------------------------------


## RecursiveCharacterTextSplitter:最常用

In [64]:
# 举例1:使用split_text()方法演示
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
text = """
LangChain是一个用于开发语言模型驱动的应用程序的框架\n\n它提供块一套工具和抽象方法\n使开发者\n
能够更容易地构建复杂的应用程序
"""
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    # add_start_index=True, #元数据中包含 chunk 的起始索引
)
paragraphs=text_splitter.split_text(text)

for paragraph in paragraphs:
    print(paragraph)
    # print('----')

LangChain
是一个用于开发语言模
型驱动的应用程序的框
架
它提供块一套工具和
抽象方法
使开发者
能够更容易地构建复
杂的应用程序


In [13]:
# 举例2:使用元数据
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
text = """LangChain是一个用于开发语言模型驱动的应用程序的框架的\n\n它提供块一套工具和抽象方法\n使开发者\n
能够更容易地构建复杂的应用程序
"""
text_list=[text]
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True, #元数据中包含 chunk 的起始索引 page_content='LangChain' metadata={'start_index': 1}
    # add_start_index=False, #page_content='LangChain'
)
paragraphs=text_splitter.create_documents(text_list)

for paragraph in paragraphs:
    print(paragraph)

page_content='LangChain是' metadata={'start_index': 0}
page_content='一个用于开发语言模型' metadata={'start_index': 10}
page_content='驱动的应用程序的框架' metadata={'start_index': 20}
page_content='的' metadata={'start_index': 30}
page_content='它提供块一套工具和' metadata={'start_index': 33}
page_content='抽象方法' metadata={'start_index': 42}
page_content='使开发者' metadata={'start_index': 47}
page_content='能够更容易地构建复' metadata={'start_index': 53}
page_content='杂的应用程序' metadata={'start_index': 62}


In [14]:
# 举例3:切分本地文件
from langchain_text_splitters import RecursiveCharacterTextSplitter

'''["\n\n", "\n", " ", ""]
    步骤1:先按\n\n拆分
    步骤2:查看是否小于100
    步骤3:大于则再按\n拆分，没有\n则按空格，再没有按字符
'''
# 打开.txt文件
with open('asset/load/08-ai.txt',encoding='utf-8') as f:
    state_of_union=f.read()

# 定义RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=80,
    # add_start_index=True,
    length_function=len
)
# 分割文本
text=text_splitter.create_documents([state_of_union])

for e in text:
    print(e.page_content)
    print('-----')


人工智能（AI）是什么？
-----
人工智能（Artificial
-----
Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
-----
指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的能力。
-----
1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
-----
深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
-----
自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。

2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。
-----
2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
-----
医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。

金融：欺诈检测、智能投顾（如风险评估模型）。
-----
交通：自动驾驶汽车通过传感器和AI算法实现安全导航。

金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。
-----
金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
-----
教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：

伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。
-----
3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：

伦理争议：数据隐私、算法偏见

In [20]:
# 举例4:使用split_documents()方法演示，利用PDFLoader加载文档，对文档内容递归切割
# 导入依赖
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader=PyPDFLoader('./asset/load/02-load.pdf')

docs=loader.load()

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
    add_start_index=True
)
text=text_splitter.split_documents(docs)

for e in text:
    print(e.page_content)


"他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不
知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。
那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老想
着远远的一辆车，可以使他自由，独立，像自己的手脚的那么一辆车。" 
 
"他吃，他喝，他嫖，他赌，他懒，他狡猾， 因为他没了心，他的心被人家摘了去。他
只剩下那个高大的肉架子，等着溃烂，预备着到乱死岗子去。……体面的、要强的、好梦想
的、利己的、个人的、健壮的、伟大的祥子，不知陪着人家送了多少回殡；不知道何时何地
会埋起他自己来， 埋起这堕落的、 自私的、 不幸的、 社会病胎里的产儿， 个人主义的末路鬼！
"


## 使用TokenTextSplitter

In [35]:
# 举例1:导入相关依赖
from langchain_text_splitters import TokenTextSplitter

text='人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自从20世纪50年代诞生以来，人工智能经历了多次起伏'
text_splitter=TokenTextSplitter(
    chunk_size=33, # 最大 token数为 33
    chunk_overlap=0,
    encoding_name='cl100k_base'  # 使用Open AI的编码器
)

texts=text_splitter.split_text(text)

for i,text in enumerate(texts):
    print(f'块{i+1}:长度{len(text)} 内容:{text}')
    print('----')


块1:长度29 内容:人工智能是一个强大的开发框架。它支持多种语言模型和工具链。
----
块2:长度31 内容:人工智能是指通过计算机程序模拟人类智能的一门科学。自从20世纪
----
块3:长度20 内容:50年代诞生以来，人工智能经历了多次起伏
----


## 使用SemanticChunker：语义分块